# Shortest Paths and Network Flow

## Overview

Shortest path algorithms find optimal routes through networks. Network flow algorithms find maximum throughput or minimum cost flows. Both have direct ecological applications: dispersal corridors, connectivity analysis, and bottleneck identification.

**Key algorithms:**

| Algorithm | Use case | Complexity |
|---|---|---|
| Dijkstra | Shortest path, non-negative weights | O(E log V) |
| Bellman-Ford | Shortest path, negative weights | O(VE) |
| Floyd-Warshall | All-pairs shortest paths | O(V³) |
| BFS | Unweighted shortest path | O(V+E) |
| Max flow (Ford-Fulkerson) | Maximum throughput | O(VE²) |
| Min cut | Bottleneck identification | Dual of max flow |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

rng = np.random.default_rng(42)

# Ecological dispersal network: nodes = habitat patches, edges = dispersal corridors
# Edge weight = dispersal resistance (inverse of connectivity)
patches = ['P01','P02','P03','P04','P05','P06','P07','P08','P09','P10']
G_disp  = nx.Graph()
G_disp.add_nodes_from(patches)
# Connectivity edges (lower resistance = better corridor)
edges = [
    ('P01','P02', 1.2), ('P01','P03', 3.5), ('P02','P04', 2.1),
    ('P03','P04', 1.8), ('P04','P05', 4.2), ('P04','P06', 2.5),
    ('P05','P07', 1.5), ('P06','P07', 3.0), ('P07','P08', 2.2),
    ('P08','P09', 1.3), ('P08','P10', 5.0), ('P09','P10', 1.7),
]
G_disp.add_weighted_edges_from(edges, weight='resistance')
pos = nx.spring_layout(G_disp, seed=42)
print(f"Dispersal network: {G_disp.number_of_nodes()} patches, {G_disp.number_of_edges()} corridors")

---
## Dijkstra Shortest Paths

In [ ]:
# Shortest dispersal path: P01 -> P10
source, target = 'P01', 'P10'
path = nx.dijkstra_path(G_disp, source, target, weight='resistance')
cost = nx.dijkstra_path_length(G_disp, source, target, weight='resistance')
print(f"Shortest path {source} -> {target}:")
print(f"  Route: {' -> '.join(path)}")
print(f"  Total resistance: {cost:.2f}")

# All shortest paths from P01
lengths = nx.single_source_dijkstra_path_length(G_disp, source, weight='resistance')
print(f"\nShortest path lengths from {source}:")
for node, dist in sorted(lengths.items(), key=lambda x: x[1]):
    print(f"  {node}: {dist:.3f}")

# Visualise shortest path
edge_colors = []
path_edges  = set(zip(path[:-1], path[1:]))
for u, v in G_disp.edges():
    if (u,v) in path_edges or (v,u) in path_edges:
        edge_colors.append('#e74c3c')
    else:
        edge_colors.append('lightgrey')
fig, ax = plt.subplots(figsize=(9,7))
nx.draw_networkx(G_disp, pos=pos, ax=ax, node_color='steelblue',
    node_size=600, font_color='white', edge_color=edge_colors, width=2.5)
edge_labels = {(u,v): f"{d['resistance']:.1f}" for u,v,d in G_disp.edges(data=True)}
nx.draw_networkx_edge_labels(G_disp, pos, edge_labels=edge_labels, font_size=8, ax=ax)
ax.set_title(f'Dispersal Network — Least-Resistance Path (red): {" → ".join(path)}')
ax.axis('off'); plt.tight_layout(); plt.show()

---
## All-Pairs Shortest Paths and Network Diameter

In [ ]:
# All-pairs shortest path matrix
apsp = dict(nx.all_pairs_dijkstra_path_length(G_disp, weight='resistance'))
nodes = list(G_disp.nodes())
dist_matrix = np.array([[apsp[u][v] for v in nodes] for u in nodes])

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(dist_matrix, cmap='YlOrRd')
ax.set_xticks(range(len(nodes))); ax.set_xticklabels(nodes, rotation=45, fontsize=9)
ax.set_yticks(range(len(nodes))); ax.set_yticklabels(nodes, fontsize=9)
plt.colorbar(im, ax=ax, label='Least-resistance distance')
ax.set_title('All-Pairs Shortest Path Matrix (resistance)')
plt.tight_layout(); plt.show()

diameter    = dist_matrix[~np.eye(len(nodes), dtype=bool)].max()
avg_path    = dist_matrix[~np.eye(len(nodes), dtype=bool)].mean()
print(f"Network diameter (max shortest path): {diameter:.3f}")
print(f"Average shortest path length:         {avg_path:.3f}")
most_isolated = nodes[np.argmax(dist_matrix.mean(axis=1))]
print(f"Most isolated patch (highest avg dist): {most_isolated}")

---
## Maximum Flow and Minimum Cut

In [ ]:
# Directed flow network: water flow through stream segments
G_flow = nx.DiGraph()
edges_flow = [
    ('SOURCE',  'S1',    10), ('SOURCE', 'S2', 12),
    ('S1',      'MID1',   8), ('S1',    'MID2', 5),
    ('S2',      'MID2',   7), ('S2',    'MID3', 9),
    ('MID1',    'SINK',   6), ('MID2',  'SINK', 9),
    ('MID3',    'SINK',   8),
]
for u, v, cap in edges_flow:
    G_flow.add_edge(u, v, capacity=cap)

# Maximum flow from SOURCE to SINK
flow_value, flow_dict = nx.maximum_flow(G_flow, 'SOURCE', 'SINK')
print(f"Maximum flow SOURCE -> SINK: {flow_value}")
print("\nFlow on each edge:")
for u in flow_dict:
    for v, f in flow_dict[u].items():
        if f > 0:
            cap = G_flow.edges[u,v]['capacity']
            print(f"  {u:8s} -> {v:8s}: {f}/{cap} (utilisation={f/cap:.0%})")

# Minimum cut
cut_value, (reachable, non_reachable) = nx.minimum_cut(G_flow, 'SOURCE', 'SINK')
print(f"\nMinimum cut value: {cut_value}  (= max flow by max-flow min-cut theorem)")
print(f"Cut edges (bottleneck segments):")
for u in reachable:
    for v in G_flow.successors(u):
        if v in non_reachable:
            print(f"  {u} -> {v}  (capacity={G_flow.edges[u,v]['capacity']})")

In [ ]:
# Critical node analysis: which patch removal most reduces connectivity?
def mean_pairwise_distance(G, weight='resistance'):
    try:
        apsp = dict(nx.all_pairs_dijkstra_path_length(G, weight=weight))
        nodes = list(G.nodes())
        dists = [apsp[u][v] for u in nodes for v in nodes if u != v]
        return np.mean(dists) if dists else np.inf
    except nx.NetworkXError:
        return np.inf

baseline_dist = mean_pairwise_distance(G_disp)
criticality   = {}
for node in G_disp.nodes():
    G_temp = G_disp.copy()
    G_temp.remove_node(node)
    if nx.is_connected(G_temp):
        d = mean_pairwise_distance(G_temp)
        criticality[node] = d - baseline_dist
    else:
        criticality[node] = np.inf   # disconnects network

crit_sorted = sorted(criticality.items(), key=lambda x: x[1], reverse=True)
print("Node criticality (removal increases avg path length by):")
for node, delta in crit_sorted:
    flag = ' *** DISCONNECTS NETWORK ***' if np.isinf(delta) else ''
    val  = 'inf' if np.isinf(delta) else f'+{delta:.3f}'
    print(f"  {node}: {val}{flag}")

---

## Common Pitfalls

**1. Calling `nx.dijkstra_path()` on a graph with negative edge weights**  
Dijkstra's algorithm requires all edge weights to be non-negative. With negative weights, the algorithm may return incorrect shortest paths silently. Use `nx.bellman_ford_path()` for graphs with negative weights, and check `(np.array(weights) < 0).any()` before choosing an algorithm.

**2. Using resistance distance and geodesic distance interchangeably**  
Geodesic (shortest-path) distance and resistance distance measure different ecological concepts. Shortest-path distance finds the single least-cost route; resistance distance averages over all paths and is more appropriate for random-walk dispersal models. Choose based on the biological movement process.

**3. Interpreting maximum flow without checking whether the graph is connected**  
If source and sink are in disconnected components, `nx.maximum_flow()` raises `NetworkXError`. Always verify connectivity with `nx.has_path(G, source, sink)` before computing flow.

**4. Removing nodes for criticality analysis without accounting for isolated subgraphs**  
Removing a bridge node may disconnect the graph, making shortest-path length infinite for disconnected pairs. Handle this explicitly — either assign a large penalty, count the number of disconnected pairs, or use algebraic connectivity as the robustness metric.

**5. Confusing edge capacity and edge weight in flow vs path algorithms**  
NetworkX uses `weight` for shortest-path algorithms and `capacity` for flow algorithms — these are different edge attributes. A graph built with `weight` but analysed with `maximum_flow()` returns zero flow because no `capacity` attribute exists. Always name edge attributes correctly for the intended algorithm.
---
*python_methods_library - Samantha McGarrigle*